# Claude with MCP: Building a Native Python MCP Client

This notebook shows how to connect **Claude** to a [Model Context Protocol (MCP)](https://modelcontextprotocol.io) server using the official `mcp` Python SDK — no third-party abstractions, just the protocol itself.

By the end you will have:
- A working MCP server that exposes three weather-related tools
- An MCP client class that launches it as a subprocess and opens a `stdio` session
- A complete **agentic loop** where Claude decides which tools to call, executes them, and synthesises a final answer

---

## Prerequisites

| Requirement | Details |
|---|---|
| Python | 3.10 or later |
| `uv` | [Install guide](https://docs.astral.sh/uv/getting-started/installation/) |
| Anthropic API key | [Get one here](https://console.anthropic.com/settings/keys) |

Store your key in a `.env` file next to this notebook:

```
ANTHROPIC_API_KEY=your-key-here
```

## 1. Setup

Install the three packages we need. `mcp` brings the full Python SDK (client + server); `anthropic` gives us the Messages API; `python-dotenv` loads the API key.

In [2]:
%%capture
%pip install mcp anthropic python-dotenv

## 2. The MCP Server

An MCP server is just a process that speaks the MCP protocol over `stdio` (or HTTP). Here we write a minimal weather server with **three tools** using `mcp.server.fastmcp.FastMCP` — the high-level decorator API that ships inside the official `mcp` package.

We use `%%writefile` to save it as a standalone script. The client will launch it as a subprocess later.

In [1]:
%%writefile weather_server.py
"""
weather_server.py — A minimal MCP server exposing three weather tools.

Run directly to test:
    python weather_server.py

The MCPClient in this notebook launches it automatically as a subprocess.
"""
import random

from mcp.server.fastmcp import FastMCP

mcp = FastMCP("WeatherServer")

# ---------------------------------------------------------------------------
# Static dataset — no external API calls needed
# ---------------------------------------------------------------------------
_CITIES: dict[str, dict] = {
    "london":   {"condition": "overcast",      "temp_c": 12, "humidity": 78},
    "tokyo":    {"condition": "sunny",          "temp_c": 24, "humidity": 55},
    "new york": {"condition": "partly cloudy",  "temp_c": 18, "humidity": 72},
    "sydney":   {"condition": "sunny",          "temp_c": 26, "humidity": 62},
    "paris":    {"condition": "light rain",     "temp_c": 14, "humidity": 83},
}


@mcp.tool()
def get_current_weather(city: str) -> str:
    """Return the current weather conditions for a given city."""
    data = _CITIES.get(city.lower().strip())
    if data is None:
        available = ", ".join(_CITIES)
        return f"No data for '{city}'. Available cities: {available}."
    return (
        f"{city.title()}: {data['condition']}, "
        f"{data['temp_c']}\u00b0C, humidity {data['humidity']}%"
    )


@mcp.tool()
def get_temperature_forecast(city: str, days: int = 3) -> str:
    """Return a simple day-by-day temperature forecast (1-7 days)."""
    days = min(max(days, 1), 7)  # clamp to [1, 7]
    data = _CITIES.get(city.lower().strip())
    if data is None:
        available = ", ".join(_CITIES)
        return f"No data for '{city}'. Available cities: {available}."
    base = data["temp_c"]
    forecast = ", ".join(
        f"Day {i}: {base + random.randint(-3, 3)}\u00b0C"
        for i in range(1, days + 1)
    )
    return f"{city.title()} {days}-day forecast: {forecast}"


@mcp.tool()
def convert_temperature(celsius: float) -> str:
    """Convert a temperature from Celsius to Fahrenheit and Kelvin."""
    fahrenheit = (celsius * 9 / 5) + 32
    kelvin = celsius + 273.15
    return f"{celsius}\u00b0C = {fahrenheit:.1f}\u00b0F = {kelvin:.2f} K"


if __name__ == "__main__":
    mcp.run()

Writing weather_server.py


## 3. The MCP Client

The client does three things:

1. **Connects** — spawns the server subprocess and opens an MCP session over `stdio`
2. **Bridges** — translates MCP tool schemas into the format the Anthropic API expects
3. **Loops** — keeps sending tool results back to Claude until it returns a plain text response

### Key imports from the `mcp` package

| Import | Role |
|---|---|
| `ClientSession` | Manages the MCP session lifecycle |
| `StdioServerParameters` | Config for launching a server subprocess |
| `stdio_client` | Async context manager that opens the subprocess and returns `(read, write)` streams |

In [ ]:
import asyncio
import os
from contextlib import AsyncExitStack
from typing import Optional

import anthropic
from dotenv import load_dotenv
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

load_dotenv()

# https://docs.anthropic.com/en/docs/about-claude/models/overview
MODEL = "claude-3-5-haiku-20241022"
SERVER_SCRIPT = "weather_server.py"


class MCPClient:
    """
    A minimal MCP client that:
      - Connects to a stdio-based MCP server
      - Exposes its tools to the Anthropic Messages API
      - Runs the tool-use loop until Claude produces a final answer
    """

    def __init__(self) -> None:
        self.session: Optional[ClientSession] = None
        # AsyncExitStack lets us enter async context managers and clean up
        # everything in one place via a single `aclose()` call.
        self.exit_stack = AsyncExitStack()
        self.anthropic = anthropic.Anthropic(
            api_key=os.environ.get("ANTHROPIC_API_KEY")
        )

    # ------------------------------------------------------------------
    # Connection
    # ------------------------------------------------------------------

    async def connect(self, server_script: str) -> None:
        """Launch the MCP server subprocess and open a session."""
        params = StdioServerParameters(
            command="python",
            args=[server_script],
        )
        # stdio_client returns (read_stream, write_stream)
        read, write = await self.exit_stack.enter_async_context(
            stdio_client(params)
        )
        self.session = await self.exit_stack.enter_async_context(
            ClientSession(read, write)
        )
        await self.session.initialize()

        tools = (await self.session.list_tools()).tools
        print(f"Connected to '{server_script}' — available tools:")
        for t in tools:
            print(f"  * {t.name}: {t.description}")

    # ------------------------------------------------------------------
    # Tool-schema bridge
    # ------------------------------------------------------------------

    async def _get_tools(self) -> list[dict]:
        """
        Fetch the current tool list from the server and convert it to
        the shape that the Anthropic Messages API expects.

        MCP's ``tool.inputSchema`` is already a JSON Schema dict, so it
        maps directly to Anthropic's ``input_schema`` field — no
        translation needed beyond renaming the key.
        """
        response = await self.session.list_tools()
        return [
            {
                "name": t.name,
                "description": t.description,
                "input_schema": t.inputSchema,  # JSON Schema -> Anthropic format
            }
            for t in response.tools
        ]

    # ------------------------------------------------------------------
    # Agentic loop
    # ------------------------------------------------------------------

    async def query(self, user_message: str) -> str:
        """
        Send a message to Claude together with the MCP server's tools.

        The loop works like this:

        1. Send the conversation to Claude.
        2. If Claude requests tool calls, execute them via the MCP session.
        3. Append the results and loop back to step 1.
        4. When Claude returns only text, return it to the caller.
        """
        tools = await self._get_tools()
        messages: list[dict] = [{"role": "user", "content": user_message}]

        while True:
            response = self.anthropic.messages.create(
                model=MODEL,
                max_tokens=1024,
                tools=tools,
                messages=messages,
            )

            # Separate text blocks from tool-use blocks
            tool_calls = [
                block for block in response.content if block.type == "tool_use"
            ]

            # No tool calls -> Claude is done; return the final text
            if not tool_calls:
                return "\n".join(
                    block.text
                    for block in response.content
                    if block.type == "text"
                )

            # Append Claude's turn (may contain both text and tool_use blocks)
            messages.append({"role": "assistant", "content": response.content})

            # Execute each tool call through the MCP session
            tool_results = []
            for call in tool_calls:
                print(f"  -> Calling tool '{call.name}' with args: {call.input}")
                result = await self.session.call_tool(call.name, call.input)
                tool_results.append(
                    {
                        "type": "tool_result",
                        "tool_use_id": call.id,
                        "content": result.content,
                    }
                )

            # Feed the results back to Claude for synthesis
            messages.append({"role": "user", "content": tool_results})

    # ------------------------------------------------------------------
    # Cleanup
    # ------------------------------------------------------------------

    async def close(self) -> None:
        """Shut down the session and terminate the server subprocess."""
        await self.exit_stack.aclose()

## 4. Run It

The two queries below exercise all three tools:

- **Query 1** hits `get_current_weather` once
- **Query 2** hits `get_temperature_forecast` and `convert_temperature` in parallel — Claude decides to call both tools in a single turn

In [3]:
async def demo() -> None:
    client = MCPClient()
    try:
        await client.connect(SERVER_SCRIPT)

        queries = [
            "What's the weather like in London right now?",
            "5-day forecast for Tokyo, and what is 24 Celsius in Fahrenheit?",
        ]

        for q in queries:
            print(f"\n{'=' * 60}")
            print(f"User: {q}")
            print("=" * 60)
            answer = await client.query(q)
            print(f"\nClaude: {answer}")

    finally:
        await client.close()


asyncio.run(demo())

Connected to 'weather_server.py' — available tools:
  * get_current_weather: Return the current weather conditions for a given city.
  * get_temperature_forecast: Return a simple day-by-day temperature forecast (1-7 days).
  * convert_temperature: Convert a temperature from Celsius to Fahrenheit and Kelvin.

User: What's the weather like in London right now?
  -> Calling tool 'get_current_weather' with args: {'city': 'London'}

Claude: Based on the current data for London, it is overcast with a temperature of 12°C and 78% humidity.

User: 5-day forecast for Tokyo, and what is 24 Celsius in Fahrenheit?
  -> Calling tool 'get_temperature_forecast' with args: {'city': 'Tokyo', 'days': 5}
  -> Calling tool 'convert_temperature' with args: {'celsius': 24.0}

Claude: Here is the information you requested:
- The 5-day forecast for Tokyo: Day 1: 26°C, Day 2: 24°C, Day 3: 25°C, Day 4: 21°C, Day 5: 23°C.
- 24.0°C is equal to 75.2°F (and 297.15 K).


## 5. How It Works

```
Notebook kernel
|
|   StdioServerParameters(command="python", args=["weather_server.py"])
|                |
|                v
|   +-------------------------+
|   |  weather_server.py      |  <- subprocess
|   |  FastMCP("WeatherServer")|
|   |  tools: get_current_weather
|   |         get_temperature_forecast
|   |         convert_temperature
|   +------------+-----------+
|           stdio (JSON-RPC)
|                |
|   ClientSession.initialize()   -> handshake
|   ClientSession.list_tools()   -> fetch schema
|                |
|   Anthropic Messages API
|   +-----------------------------------------+
|   | messages.create(model, tools, messages) |
|   |   stop_reason == "tool_use"             |
|   |     -> session.call_tool(name, args)    |
|   |     -> append tool_result to messages   |
|   |   stop_reason == "end_turn"             |
|   |     -> return final text                |
|   +-----------------------------------------+
```

### Why `inputSchema` maps directly

MCP tool schemas are vanilla [JSON Schema](https://json-schema.org/) objects — the same format the Anthropic API expects for `input_schema`. The bridge in `_get_tools()` is a single key rename:

```python
# MCP property   ->  Anthropic parameter
tool.name        ->  "name"
tool.description ->  "description"
tool.inputSchema ->  "input_schema"   # same JSON Schema, different key name
```

### The agentic loop

The `while True` in `MCPClient.query()` is the **tool-use loop**:

1. Claude receives the user message + tool definitions
2. If it wants to call tools, it responds with `stop_reason="tool_use"` and one or more `tool_use` content blocks
3. The client executes each call via `session.call_tool()` and appends a `tool_result` block
4. Claude receives the results and either calls more tools or produces a final `end_turn` response

This loop is identical whether you have 1 tool or 100 — the MCP server handles discovery dynamically.

## Next Steps

- **Add real tools** — replace the mock data with live API calls (e.g., [Open-Meteo](https://open-meteo.com/) for weather, a database query, a filesystem reader)
- **Use resources** — MCP servers can also expose *resources* (read-only data) via `@mcp.resource()`. Add `session.list_resources()` and inject them into the system prompt
- **Try remote transport** — swap `stdio_client` for `streamablehttp_client` to connect to a server running on a different machine

### References

- [MCP Python SDK](https://github.com/modelcontextprotocol/python-sdk)
- [Build an MCP client — official guide](https://modelcontextprotocol.io/docs/develop/build-client)
- [Anthropic tool use docs](https://docs.anthropic.com/en/docs/agents-and-tools/tool-use/overview)
- [Claude models overview](https://docs.anthropic.com/en/docs/about-claude/models/overview)